# Volume 14, workshop chapter 4: Selberg's theorem at many points (Python companion)

The covariance of Lemma 4.1 computed exactly, the joint moments of Lemma 4.3 checked on actual averages over $t$, the zeta function itself at two nearby points (Figure 4.1), and the maximum on a short interval (Remark 4.4). The computer-science application is **evaluating many trigonometric sums at once with matrix products**, and **computing $\zeta(\frac12+it)$ at height $10^6$** with `mpmath` (which uses the Riemann--Siegel formula there). Exercises first, solutions at the end. Requires `numpy` and `mpmath`.

In [1]:
import math, numpy as np, mpmath
rng = np.random.default_rng(4)
def primes_upto(x):
    s = np.ones(x + 1, bool); s[:2] = False
    for i in range(2, int(x**0.5) + 1):
        if s[i]: s[i*i::i] = False
    return np.nonzero(s)[0].astype(float)

## Exercises

**P4.1 (★) The covariance.** For $x=10^6$, compute $c(h)=\sum_{p\le x}\frac{\cos(h\log p)}p$ for $h$ from $10^{-4}$ to $1$, and compare with $\min(\log\frac1h,\log\log x)$. How large is the difference?

**P4.2 (★★) Joint moments.** Let $Y(t)=\sum_{p\le 10^4}\frac{\cos(t\log p)}{\sqrt p}$. For $t$ uniform on $[T,2T]$ with $T=10^{9}$ and a shift $h$, estimate $\mathbb E[Y(t)Y(t+h)]$ and $\mathbb E[Y(t)^2Y(t+h)^2]$, and compare with $\frac12c(h)$ and with Isserlis's prediction $\frac14c(0)^2+2\cdot\frac14c(h)^2$ (Lemma 4.3).

**P4.3 (★★) The zeta function at two points.** For $300$ random $t\in[10^6,2\cdot10^6]$ compute $\log|\zeta(\frac12+it)|$ and $\log|\zeta(\frac12+it+ih)|$ for a few $h$, their correlation, and the variance of the increment (Exercise 4.4).

**P4.4 (★) Ultrametricity.** Check numerically that $c(h)$ satisfies $c(h_1-h_3)\ge\min(c(h_1-h_2),c(h_2-h_3))-O(1)$ on random triples (Exercise 4.2).

**P4.5 (★★) The maximum.** For $50$ random $t$ near $10^8$, compute $\max_{|h|\le1}Y(t+h)$ on a fine grid, with $Y$ the prime sum over $p\le10^5$, and compare with the branching prediction $\log\log x-\frac34\log\log\log x$ (Remark 4.4) and with the maximum of independent Gaussians.

## Solutions

**P4.1.** One vectorised sum per $h$.

In [2]:
pr = primes_upto(10**6); lp = np.log(pr)
L = math.log(math.log(1e6))
print(f'log log x = {L:.3f}')
for h in [1e-4, 1e-3, 1e-2, 0.03, 0.1, 0.3, 1.0]:
    c = float(np.sum(np.cos(h * lp) / pr))
    print(f'h = {h:<7g} c(h) = {c:6.3f}   min(log 1/h, log log x) = {min(math.log(1/h), L):6.3f}   difference {c - min(math.log(1/h), L):+.3f}')

log log x = 2.626
h = 0.0001  c(h) =  2.887   min(log 1/h, log log x) =  2.626   difference +0.262
h = 0.001   c(h) =  2.887   min(log 1/h, log log x) =  2.626   difference +0.261
h = 0.01    c(h) =  2.883   min(log 1/h, log log x) =  2.626   difference +0.257
h = 0.03    c(h) =  2.846   min(log 1/h, log log x) =  2.626   difference +0.220
h = 0.1     c(h) =  2.459   min(log 1/h, log log x) =  2.303   difference +0.156
h = 0.3     c(h) =  0.825   min(log 1/h, log log x) =  1.204   difference -0.379
h = 1       c(h) =  0.240   min(log 1/h, log log x) =  0.000   difference +0.240


The difference stays bounded, between about $-0.4$ and $+0.3$ (the constant $M=0.2615$ of Mertens appears at small $h$), as Lemma 4.1 says; here there is no damping $p^{-\delta}$, so the error $O(\log W)$ of the lemma is replaced by $O(1)$.

**P4.2.** The sums over $p$ for many values of $t$ at once are a matrix product: $\cos(t\log p)$ for a vector of $t$'s and a vector of $p$'s.

In [3]:
pr4 = primes_upto(10**4); lp4 = np.log(pr4); w4 = pr4**-0.5
def Y(ts):
    out = np.empty(len(ts))
    for i in range(0, len(ts), 2000):
        out[i:i+2000] = np.cos(np.outer(ts[i:i+2000], lp4)) @ w4
    return out
T = 1e9; ts = rng.uniform(T, 2*T, 40000)
c0 = float(np.sum(1 / pr4))
for h in [0.0, 0.01, 0.1, 0.5]:
    ch = float(np.sum(np.cos(h * lp4) / pr4))
    a, b = Y(ts), Y(ts + h)
    print(f'h={h:<5}: E[YY]={np.mean(a*b):.3f} (pred {ch/2:.3f})   E[Y^2 Y^2]={np.mean(a*a*b*b):.3f} (Isserlis {c0**2/4 + 2*ch**2/4:.3f})')

h=0.0  : E[YY]=1.241 (pred 1.242)   E[Y^2 Y^2]=4.481 (Isserlis 4.624)


h=0.01 : E[YY]=1.240 (pred 1.241)   E[Y^2 Y^2]=4.476 (Isserlis 4.619)


h=0.1  : E[YY]=1.143 (pred 1.145)   E[Y^2 Y^2]=4.010 (Isserlis 4.164)


h=0.5  : E[YY]=0.206 (pred 0.201)   E[Y^2 Y^2]=1.477 (Isserlis 1.622)


The covariances agree to within $0.005$. The fourth moments are about $0.14$ below Isserlis's formula, for every $h$: that is the term $O(v_T^{K/2-2})$ of Lemma 4.3, from repeated primes (here $K=4$, so it is $O(1)$, against a main term of about $4.6$ at $h=0$), negative because $\mathbb E\cos^4=\frac38<3\,(\mathbb E\cos^2)^2=\frac34$.

**P4.3.** `mpmath.zeta` at height $10^6$ takes about $15$ ms.

In [4]:
def lz(t): return float(mpmath.log(abs(mpmath.zeta(mpmath.mpf(0.5) + 1j * mpmath.mpf(t)))))
ts3 = rng.uniform(1e6, 2e6, 300)
base = np.array([lz(t) for t in ts3])
print(f'variance of log|zeta| = {base.var():.3f}   (1/2) log log T = {0.5*math.log(math.log(1.5e6)):.3f}')
for h in [0.003, 0.03, 0.1, 0.3, 1.0]:
    sh = np.array([lz(t + h) for t in ts3])
    print(f'h={h:<6}: correlation {np.corrcoef(base, sh)[0,1]:.3f}   variance of increment {np.var(sh - base):.3f}')

variance of log|zeta| = 2.059   (1/2) log log T = 1.327


h=0.003 : correlation 0.992   variance of increment 0.035


h=0.03  : correlation 0.821   variance of increment 0.838


h=0.1   : correlation 0.645   variance of increment 1.396


h=0.3   : correlation 0.254   variance of increment 3.006


h=1.0   : correlation 0.093   variance of increment 3.650


The variance is about $2$, far above $\frac12\log\log T\approx1.3$: the constant of the Keating--Snaith prediction discussed in Volume 14, Chapter 7. The correlation falls from nearly $1$ to about $0.1$ as $h$ goes from $10^{-3}$ to $1$, and the variance of the increment grows like the part of the variance that the two points do not share, $2(1-\text{correlation})\times$ the variance, as in Exercise 4.4.

**P4.4.** Random triples in $[0,1]$.

In [5]:
pr5 = primes_upto(10**5); lp5 = np.log(pr5)
cf = lambda h: float(np.sum(np.cos(h * lp5) / pr5))
worst = 0.0
for _ in range(300):
    h1, h2, h3 = rng.uniform(0, 1, 3) * 10**rng.uniform(-4, 0)
    worst = min(worst, cf(h1 - h3) - min(cf(h1 - h2), cf(h2 - h3)))
print(f'smallest value of c(h1-h3) - min(c(h1-h2), c(h2-h3)) over 300 triples: {worst:.3f}')

smallest value of c(h1-h3) - min(c(h1-h2), c(h2-h3)) over 300 triples: -0.881


The defect never goes below about $-1$: up to a bounded error, the covariance is ultrametric, the signature of a tree.

**P4.5.** A grid of spacing $0.002$ is finer than the scale $\frac1{\log x}$ on which the prime sum over $p\le10^5$ varies.

In [6]:
w5 = pr5**-0.5
grid = np.arange(-1, 1, 0.002)
mx = []
for t in rng.uniform(1e8, 2e8, 50):
    vals = np.cos(np.outer(t + grid, lp5)) @ w5
    mx.append(vals.max())
mx = np.array(mx)
L2 = math.log(math.log(1e5)); L3 = math.log(L2)
var = 0.5 * float(np.sum(1 / pr5))
n_ind = 2 * math.log(1e5)
iid = math.sqrt(2 * var * math.log(n_ind))
print(f'mean maximum {mx.mean():.3f} (sd {mx.std():.3f})')
print(f'branching prediction L2 - (3/4) L3 = {L2 - 0.75*L3:.3f};   max of {n_ind:.0f} independent N(0,{var:.2f}): about {iid:.3f}')

mean maximum 1.739 (sd 0.849)
branching prediction L2 - (3/4) L3 = 1.773;   max of 23 independent N(0,1.35): about 2.913


The mean maximum, about $1.7$ with a standard deviation near $0.85$, is well below the $2.9$ that $23$ independent Gaussians of the same variance would give (one per spacing $\frac1{\log x}$): the correlations lower the maximum, as the $-\frac34$ (against $-\frac14$ for independent variables) predicts. The closeness to $\log\log x-\frac34\log\log\log x=1.77$ is partly luck: at $x=10^5$ the $O(1)$ of the conjecture is as large as the terms we compare.